# Õppetund 08 - Mitme agendi disainimuster


## Seadistamine


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Miks mitmeagendisüsteemid?

Reaalsed ülesanded nagu reisi planeerimine hõlmavad palju erinevaid valdkondi — logistikat, kohalikku tundmist, eelarvestamist ja muud. Üks agent, kes üritab kõike ise korda ajada, muutub kiiresti raskehallatavaks.

Mitmeagendisüsteemid lahendavad selle **spetsialiseerumise** abil: iga agent keskendub ühele valdkonnale, tootes üldistajast kvaliteetsemaid tulemusi. Samuti parandavad nad **skaalautuvust** — saate lisada uusi agente (nt lennuekspert, restorani kriitik) ilma olemasolevat töövoogu ümber kirjutamata. Agendid töötavad koos struktureeritud torujuhtme kaudu, edastades konteksti ühest teiseni.


## Spetsialiseeritud agentide loomine


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Jada töövoo loomine

`WorkflowBuilder` võimaldab teil ühendada agente suunatud graafikuks. Siin loome lihtsa kahest etapist koosneva torujuhtme: **TravelPlanner** koostab marsruudi kavandi ja seejärel **TravelConcierge** vaatab selle üle ning täiustab seda.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Rohkem agentide lisamine töövoogu

Üks suurimaid eeliseid multiagentide mustri juures on selle lihtsus laiendamisel. Allpool lisame **BudgetReviewer** agendi, kes kontrollib plaani reisi eelarvega, märkides esemed, mis võivad kulud üle piiri viia, ning soovitab raha säästvaid alternatiive. Töövoog käivitab nüüd järjestikku kolm agenti:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kokkuvõte

Selles õppetükis õppisite, kuidas:

1. **Luuakse spetsialiseeritud agente** — igaühel on kindel roll (planeerimine, teenindus, eelarve ülevaatus).
2. **Ühendage agentidele järjestikune töövoog** kasutades `WorkflowBuilder` ja `add_edge`.
3. **Voogedastage väljund** mitmeagendi torustikust, jälgides, milline agent parasjagu räägib.
4. **Laiendage töövoogu** lisades ahelasse uusi agente ilma olemasolevaid muutmata.

Mitmeagendi disainimuster hoiab iga agendi lihtsana, võimaldades samal ajal toota rikkalikumaid ja põhjalikumalt üle vaadatud tulemusi, kui üks agent suudaks üksinda saavutada.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:
See dokument on tõlgitud tehisintellekti tõlketeenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi püüame tagada täpsust, tuleb arvestada, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument oma emakeeles tuleks pidada autoriteetseks allikaks. Tähtsa teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta tõlgendamisest või arusaamatustest, mis võivad sellest tõlkest tekkida.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
